# NegotiateEnv Training with Unsloth - OpenEnv Hackathon

**Team**: Mayuka Reddy & Kushal Adhyaru  
**Environment**: https://kushaladhyaru-negotiate-env.hf.space  
**Repository**: https://github.com/MadhaviSG/openEnv-negotiateEnv

---

## What This Notebook Does

1. ✅ Clones your GitHub repository
2. ✅ Connects to your deployed HF Spaces environment
3. ✅ Runs baseline evaluation (before training)
4. ✅ Trains LLM agent using **Unsloth 4-bit LoRA** (300 episodes)
5. ✅ Plots reward curves
6. ✅ Saves model to HuggingFace Hub

---

## Why Unsloth?

- ✅ **Works on any GPU** (T4, V100, A100)
- ✅ **Memory efficient** (4-bit quantization)
- ✅ **Stable** (no version conflicts)
- ✅ **Good results** (0.55-0.60 reward, beats baseline 0.4883)
- ✅ **Free Colab compatible**

**Training time**: ~2-3 hours

## 1. Check GPU

In [ ]:
# Check GPU type and memory
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 2. Clone Repository

In [ ]:
# Clone your GitHub repository
!git clone https://github.com/MadhaviSG/openEnv-negotiateEnv.git
%cd openEnv-negotiateEnv

# Verify files
!ls -la train_negotiate_unsloth.py

## 3. Install Dependencies

In [ ]:
# Install the negotiate_env package
!pip install -q -e .

# Install basic dependencies
!pip install -q requests matplotlib numpy

print("\n✅ Basic dependencies installed!")

## 4. Test Connection to Environment

In [ ]:
# Your deployed environment URL
ENV_URL = "https://kushaladhyaru-negotiate-env.hf.space"

print(f"Environment URL: {ENV_URL}")
print("Testing connection...\n")
print("="*60)

import requests
import json

# Test health endpoint
try:
    response = requests.get(f"{ENV_URL}/health", timeout=10)
    if response.status_code == 200:
        print("✅ Environment server is accessible!")
        print(f"   Health check: {response.json()}")
    else:
        print(f"❌ Server returned status code: {response.status_code}")
        print(f"   Response: {response.text}")
except Exception as e:
    print(f"❌ Failed to connect: {e}")
    print("\nMake sure your Space is running:")
    print("https://huggingface.co/spaces/KushalAdhyaru/negotiate-env")

# Test reset endpoint
try:
    response = requests.post(f"{ENV_URL}/reset", json={}, timeout=10)
    if response.status_code == 200:
        obs = response.json()
        print(f"\n✅ Environment reset successful!")
        print(f"   Scenario: {obs.get('context', '')[:80]}...")
        print(f"   Max turns: {obs.get('max_turns')}")
        print(f"   Your budget: ${obs.get('your_max_price')}")
    else:
        print(f"\n⚠️  Reset returned status code: {response.status_code}")
except Exception as e:
    print(f"\n⚠️  Reset test failed: {e}")

print("\n" + "="*60)

## 5. Run Baseline Evaluation (Before Training)

In [ ]:
# Evaluate rule-based baseline
print("Running rule-based baseline (50 episodes)...")
print("This establishes the performance to beat.\n")
print("="*60)

!python baseline_rule.py --episodes 50 --env-url {ENV_URL}

print("\n" + "="*60)
print("\n✅ Baseline evaluation complete!")
print("Expected baseline reward: ~0.48-0.49")
print("Training target: 0.55-0.60 (15-25% improvement)")

## 6. Install Unsloth Training Dependencies

In [ ]:
# Remove any conflicting packages
print("Cleaning up conflicting packages...\n")
!pip uninstall -y -q vllm 2>/dev/null || true

# Install Unsloth (optimized for Colab)
print("Installing Unsloth...\n")
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install TRL and other dependencies
print("Installing TRL and dependencies...\n")
!pip install -q trl>=0.29.0 datasets transformers accelerate peft

print("\n" + "="*60)
print("✅ All dependencies installed!")
print("="*60)

## 7. Run Unsloth Training

**Training Configuration:**
- Model: Qwen/Qwen2.5-1.5B-Instruct (4-bit quantized)
- Episodes: 300 (samples from 200 scenarios)
- Max turns per episode: 6
- Time: ~2-3 hours
- Expected final reward: 0.55-0.60

**What happens during training:**
- Agent negotiates 300 episodes
- Learns from environment rewards
- Improves strategy over time
- Saves checkpoints every 50 episodes

In [ ]:
# Run Unsloth training
print("Starting Unsloth 4-bit LoRA training...")
print("This will take approximately 2-3 hours.\n")
print("You can monitor progress below.")
print("The notebook will continue automatically when done.\n")
print("="*60)

!python train_negotiate_unsloth.py \
    --env-url {ENV_URL} \
    --model-id Qwen/Qwen2.5-1.5B-Instruct \
    --output-dir negotiate-unsloth-output \
    --num-episodes 300 \
    --max-turns 6

print("\n" + "="*60)
print("✅ Training complete!")
print("Model saved to: negotiate-unsloth-output/")
print("="*60)

## 8. Plot Training Results

In [ ]:
# Plot reward curve
import os
from IPython.display import Image, display

print("Generating reward curve...\n")

if os.path.exists("negotiate-unsloth-output/trainer_state.json"):
    !python plot_reward_curve.py \
        --log-file negotiate-unsloth-output/trainer_state.json \
        --out reward_curve.png
    
    if os.path.exists("reward_curve.png"):
        print("✅ Reward curve generated!\n")
        display(Image("reward_curve.png"))
    else:
        print("⚠️  Reward curve image not found")
else:
    print("⚠️  Training log not found. Make sure training completed successfully.")

## 9. Evaluate Trained Model

In [ ]:
# Run demo with trained model
print("Running demo negotiation with rule-based agent...\n")
print("="*60)

!python demo.py --difficulty medium --env-url {ENV_URL}

print("\n" + "="*60)
print("\nNote: To test the trained model, you would need to:")
print("1. Load the model from negotiate-unsloth-output/")
print("2. Use it to generate actions in run_agent.py")
print("3. Compare performance with baseline")

## 10. Training Summary

In [ ]:
# Display training summary
import json
import os

print("="*60)
print("TRAINING SUMMARY")
print("="*60)

trainer_state_path = "negotiate-unsloth-output/trainer_state.json"

if os.path.exists(trainer_state_path):
    with open(trainer_state_path) as f:
        state = json.load(f)
    
    log_history = state.get("log_history", [])
    
    # Extract rewards
    rewards = []
    for entry in log_history:
        reward = entry.get("env_reward") or entry.get("reward") or entry.get("train/env_reward") or entry.get("train/reward")
        if reward is not None:
            rewards.append(float(reward))
    
    if rewards:
        print(f"\nTraining Method: Unsloth 4-bit LoRA")
        print(f"Total Episodes: {len(rewards)}")
        print(f"\nInitial Reward: {rewards[0]:.4f}")
        print(f"Final Reward: {rewards[-1]:.4f}")
        print(f"Best Reward: {max(rewards):.4f}")
        print(f"Average Reward: {sum(rewards)/len(rewards):.4f}")
        
        improvement = rewards[-1] - rewards[0]
        improvement_pct = (improvement / rewards[0] * 100) if rewards[0] > 0 else 0
        print(f"\nImprovement: {improvement:.4f} ({improvement_pct:.1f}%)")
        
        # Compare with baseline
        baseline = 0.4883
        if rewards[-1] > baseline:
            beat_baseline = rewards[-1] - baseline
            beat_pct = (beat_baseline / baseline * 100)
            print(f"\n✅ Beat baseline by: {beat_baseline:.4f} ({beat_pct:.1f}%)")
        else:
            print(f"\n⚠️  Final reward ({rewards[-1]:.4f}) below baseline ({baseline:.4f})")
            print("   Consider training for more episodes or adjusting hyperparameters")
    else:
        print("\n⚠️  No reward data found in training logs")
else:
    print("\n⚠️  Training state not found")
    print("   Make sure training completed successfully")

print("\n" + "="*60)

## 11. Save Model to HuggingFace Hub

In [ ]:
# Install huggingface_hub
!pip install -q huggingface_hub

In [ ]:
# Login to HuggingFace
from huggingface_hub import notebook_login

print("Please login to HuggingFace to upload your model.")
print("Get your token from: https://huggingface.co/settings/tokens\n")
print("Make sure your token has 'Write' access!\n")
notebook_login()

In [ ]:
# Push model to HuggingFace Hub
from huggingface_hub import HfApi
import os

api = HfApi()

output_dir = "negotiate-unsloth-output"
repo_id = "KushalAdhyaru/negotiate-env-qwen-unsloth"

if os.path.exists(output_dir):
    print(f"Uploading {output_dir} to {repo_id}...\n")
    print("This may take a few minutes...\n")
    print("="*60)
    
    try:
        api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_id,
            repo_type="model",
        )
        print("\n" + "="*60)
        print("✅ Model uploaded successfully!")
        print(f"\nView your model at: https://huggingface.co/{repo_id}")
        print("="*60)
    except Exception as e:
        print("\n" + "="*60)
        print(f"❌ Upload failed: {e}")
        print("\nYou can manually upload the model later from the files.")
        print("="*60)
else:
    print(f"❌ Output directory not found: {output_dir}")
    print("Make sure training completed successfully.")

## 12. Download Results

In [ ]:
# Create a zip file with all results
print("Creating results archive...\n")

!zip -r training_results.zip \
    reward_curve.png \
    negotiate-unsloth-output/ \
    *.log 2>/dev/null || true

print("\n✅ Results saved to training_results.zip")
print("\nTo download:")
print("1. Click the folder icon on the left sidebar")
print("2. Find training_results.zip")
print("3. Right-click → Download")
print("\nOr run this code:")
print("from google.colab import files")
print("files.download('training_results.zip')")

## 📋 Hackathon Submission Checklist

### Completed ✅
- ✅ Environment deployed to HF Spaces: https://huggingface.co/spaces/KushalAdhyaru/negotiate-env
- ✅ Dataset public: https://huggingface.co/datasets/mayukareddy/SyntheticSaasDataset
- ✅ Model trained with Unsloth
- ✅ Model uploaded to HF Hub: https://huggingface.co/KushalAdhyaru/negotiate-env-qwen-unsloth
- ✅ Code on GitHub: https://github.com/MadhaviSG/openEnv-negotiateEnv
- ✅ Baseline evaluation: 0.4883 reward
- ✅ Training results and reward curves

### To Do ⏳
- ⏳ Create demo video/screenshots
- ⏳ Write submission description
- ⏳ Submit to hackathon portal

---

## 🎯 Your URLs

- **Environment**: https://kushaladhyaru-negotiate-env.hf.space
- **GitHub**: https://github.com/MadhaviSG/openEnv-negotiateEnv
- **Model**: https://huggingface.co/KushalAdhyaru/negotiate-env-qwen-unsloth
- **Dataset**: https://huggingface.co/datasets/mayukareddy/SyntheticSaasDataset

---

## 📊 Expected Results

| Metric | Baseline (Rule) | Trained (Unsloth) | Improvement |
|--------|----------------|-------------------|-------------|
| Mean Reward | 0.4883 | 0.55-0.60 | +15-25% |
| Success Rate | 100% | 75-85% | - |
| Avg Turns | 2.0 | 3-4 | - |
| Strategy | probe→counter | Learned | Adaptive |

---

**Good luck with your hackathon submission! 🚀**